In [3]:
import cv2
from pathlib import Path

# Configuration
config = {
    'scale_factor': 4,  # Downsampling factor (4x means HR will be 4 times larger than LR)
    'max_images': None,  # None = all images, or specify a number like 100
    'hr_path': 'HR',  # Input HR directory
    'lr_path': 'LR/x4',  # Output LR directory
    'image_extensions': ['.jpg', '.jpeg', '.png', '.bmp']
}

def generate_lr_from_hr(config):
    """Read images from HR path and generate LR images using bicubic downsampling"""

    hr_path = Path(config['hr_path'])
    lr_path = Path(config['lr_path'])

    # Create LR output directory if it doesn't exist
    lr_path.mkdir(parents=True, exist_ok=True)

    # Get all image files directly from the HR directory (no subdirectories)
    image_files = []
    for ext in config['image_extensions']:
        image_files.extend(list(hr_path.glob(f"*{ext}")))

    image_files = sorted(image_files)

    print(f"Found {len(image_files)} images in HR directory: {hr_path}")

    # Limit total images if specified
    if config['max_images'] is not None:
        image_files = image_files[:config['max_images']]
        print(f"Processing first {len(image_files)} images...")

    total_images = 0

    for img_file in image_files:
        # Read HR image
        hr_img = cv2.imread(str(img_file))

        if hr_img is None:
            print(f"Warning: Could not read {img_file}")
            continue

        # Use original filename for LR image with scale factor before extension
        lr_filename = f"{img_file.stem}x{config['scale_factor']}{img_file.suffix}"

        # Calculate LR dimensions
        h, w = hr_img.shape[:2]
        lr_h = h // config['scale_factor']
        lr_w = w // config['scale_factor']

        # Generate LR image using bicubic interpolation
        lr_img = cv2.resize(hr_img, (lr_w, lr_h), interpolation=cv2.INTER_CUBIC)

        # Save LR image to output directory
        lr_output_file = lr_path / lr_filename
        cv2.imwrite(str(lr_output_file), lr_img)

        total_images += 1

        if total_images % 100 == 0:
            print(f"Processed {total_images} images...")

    print(f"\nLR dataset generation complete!")
    print(f"Total images: {total_images}")
    print(f"HR source directory: {hr_path}")
    print(f"LR output directory: {lr_path}")

# Execute the generation
generate_lr_from_hr(config)


Found 5 images in HR directory: HR

LR dataset generation complete!
Total images: 5
HR source directory: HR
LR output directory: LR\x4
